# Smart Inhalo — End-to-End Dataset Pipeline

**Goal:** Build a clean, cycle-level respiratory dataset for **Correct / Wrong breathing technique** classification.

**Source:** PhysioNet *Respiratory Dataset* (PEEP study) — 80 subjects with continuous flow, pressure, tidal volume, chest and abdomen belt signals.

**Pipeline overview:**

```
PhysioNet raw signals
        ↓
Subject metadata (Trial Classification)
        ↓
Inspiratory peak detection → breathing cycles
        ↓
Feature engineering (flow / pressure / volume / chest / abdomen)
        ↓
Quality control (remove non-physiological cycles)
        ↓
Technical Correct / Wrong labels (IQR-based abnormality score)
        ↓
Smart_Inhalo_Master_Dataset.csv  (ready for ML)
```

This notebook is self-contained and presentation-ready.  
It produces the exact dataset used by the Smart Inhalo hybrid ML + DL classifier.

## 0. Setup & libraries

In [ ]:
import os
import warnings
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries loaded successfully.")
print("Working directory:", os.getcwd())

## 1. Data sources

| Source | Role |
|--------|------|
| **PhysioNet Respiratory Dataset** | Continuous multi-channel breathing signals (flow, pressure, volume, chest, abdomen) |
| **subject-info.csv** | Demographics + Trial Classification (Normal / Asthmatic / Smoker / Vaper × Sex) |
| **Processed_Dataset/** | Per-subject CSV files with pre-marked inspiratory indices |

We only use PhysioNet for the final master table.  
OMuSense / Mendeley were explored earlier but are not required for this pipeline.

In [ ]:
sources = {
    "PhysioNet": "https://physionet.org/content/respiratory-dataset/1.0.0/",
    "Subject info": "subject-info.csv (demographics + Trial Classification)",
    "Signals": "Processed_Dataset/ProcessedData_SubjectXX.csv (flow, pressure, volume, belts)",
}

for name, desc in sources.items():
    print(f"• {name:14s} → {desc}")

## 2. Load subject metadata

Subject-level attributes (sex, age, asthma, smoking/vaping status, **Trial Classification**).  
Trial Classification is later merged onto every breathing cycle.

In [ ]:
SUBJECT_INFO_URL = (
    "https://physionet.org/files/respiratory-dataset/1.0.0/subject-info.csv"
)

# PhysioNet serves this file as an Excel workbook despite the .csv extension
from io import BytesIO

response = requests.get(SUBJECT_INFO_URL, timeout=60)
response.raise_for_status()

df_info = pd.read_excel(BytesIO(response.content))
df_info.columns = df_info.columns.str.strip()

print("Subject info shape:", df_info.shape)
print("Unique subjects:", df_info["Subject Number"].nunique())
print("\nColumns:")
print(df_info.columns.tolist())
print("\nTrial Classification distribution:")
print(df_info["Trial Classification"].value_counts())
df_info.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Trial Classification counts
tc = df_info["Trial Classification"].value_counts()
axes[0].barh(tc.index, tc.values, color="#5B8DEF")
axes[0].set_xlabel("Number of subjects")
axes[0].set_title("Trial Classification (subject-level)")
for i, v in enumerate(tc.values):
    axes[0].text(v + 0.3, i, str(v), va="center", fontweight="bold")

# Sex distribution
sex = df_info["Sex (M/F)"].value_counts()
axes[1].pie(sex.values, labels=sex.index, autopct="%1.0f%%",
            colors=["#6C8FD9", "#D9A46C"], startangle=90)
axes[1].set_title("Sex distribution")

plt.tight_layout()
plt.savefig("fig_subject_demographics.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Cycle extraction & feature engineering

### Logic
1. Each subject file already contains an **Inspiratory Indicies** column (non-zero at the start of inspiration).
2. Consecutive inspiratory markers define one **breathing cycle**.
3. From every cycle we compute summary statistics of:
   - **Flow** (peak, min, mean, range, std)
   - **Pressure** (same)
   - **Tidal volume** (max, mean)
   - **Chest & abdomen** belt ranges
   - **Cycle duration**

### Quality-control rules (remove non-physiological cycles)
- Duration < 1 s or > 10 s
- Flow range < 0.5 L/s
- Pressure range < 0.5 cmH₂O
- Both chest **and** abdomen range == 0 (sensor failure)

These rules were validated on Subjects 01–03 before running on all 80 subjects.

In [ ]:
BASE_URL = (
    "https://physionet.org/files/"
    "respiratory-dataset/1.0.0/Processed_Dataset/"
)
TOTAL_SUBJECTS = 80


def quality_control(df: pd.DataFrame):
    """Remove cycles that are physiologically implausible."""
    df = df.copy()
    suspicious = (
        (df["Cycle_Duration_s"] < 1.0)
        | (df["Cycle_Duration_s"] > 10.0)
        | (df["Flow_Range_L_s"] < 0.5)
        | (df["Pressure_Range_cmH2O"] < 0.5)
        | ((df["Chest_Range_mm"] == 0) & (df["Abd_Range_mm"] == 0))
    )
    return df[~suspicious].copy(), df[suspicious].copy()


def process_subject(subject_number: int):
    """
    Download one subject, extract cycle-level features, apply QC.
    Returns (df_clean, df_removed, n_raw_rows).
    """
    filename = f"ProcessedData_Subject{subject_number:02d}.csv"
    url = BASE_URL + filename

    response = requests.get(url, timeout=90)
    response.raise_for_status()
    df_subject = pd.read_csv(StringIO(response.text))
    n_raw = len(df_subject)

    # --- inspiratory event positions ---
    event_indices = (
        df_subject.loc[
            df_subject["Inspiratory Indicies"] != 0,
            "Inspiratory Indicies",
        ]
        .astype(int)
        .values
    )
    event_indices = event_indices[
        (event_indices >= 0) & (event_indices < len(df_subject))
    ]
    event_indices = np.unique(event_indices)

    # --- feature extraction per cycle ---
    features = []
    for i in range(len(event_indices) - 1):
        start, end = event_indices[i], event_indices[i + 1]
        if end <= start:
            continue
        cycle = df_subject.iloc[start:end]
        if len(cycle) < 10:
            continue

        flow = cycle["Flow [L/s]"]
        pressure = cycle["Pressure [cmH2O]"]
        volume = cycle["V_tidal [L]"]
        chest = cycle["Chest [mm]"]
        abdomen = cycle["Abd [mm]"]

        start_time = cycle["Time [s]"].iloc[0]
        end_time = cycle["Time [s]"].iloc[-1]

        features.append({
            "Subject_ID": subject_number,
            "Cycle": i + 1,
            "Cycle_Duration_s": end_time - start_time,
            "Peak_Flow_L_s": flow.max(),
            "Min_Flow_L_s": flow.min(),
            "Mean_Flow_L_s": flow.mean(),
            "Flow_Range_L_s": flow.max() - flow.min(),
            "Flow_Std_L_s": flow.std(),
            "Peak_Pressure_cmH2O": pressure.max(),
            "Min_Pressure_cmH2O": pressure.min(),
            "Mean_Pressure_cmH2O": pressure.mean(),
            "Pressure_Range_cmH2O": pressure.max() - pressure.min(),
            "Pressure_Std_cmH2O": pressure.std(),
            "Max_Tidal_Volume_L": volume.max(),
            "Mean_Tidal_Volume_L": volume.mean(),
            "Chest_Range_mm": chest.max() - chest.min(),
            "Abd_Range_mm": abdomen.max() - abdomen.min(),
        })

    df_features = pd.DataFrame(features)
    if len(df_features) == 0:
        return pd.DataFrame(), pd.DataFrame(), n_raw

    df_clean, df_removed = quality_control(df_features)
    return df_clean, df_removed, n_raw


print("process_subject() and quality_control() defined.")

### 3.1 Run on all 80 subjects

> **Tip for presentations / offline use:**  
> If you already have `respiratory_master_dataset.csv` (the 18-column intermediate file),  
> set `USE_CACHED_MASTER = True` below to skip the lengthy download.  
> In Google Colab the notebook will also offer a file-upload prompt if the file is missing.


In [ ]:
USE_CACHED_MASTER = True   # ← set False to re-download all 80 subjects from PhysioNet
CACHED_PATH = "respiratory_master_dataset.csv"

if USE_CACHED_MASTER and os.path.exists(CACHED_PATH):
    df_master = pd.read_csv(CACHED_PATH)
    print(f"Loaded cached master from '{CACHED_PATH}'")
    print("Shape:", df_master.shape)
    print("Subjects:", df_master["Subject_ID"].nunique())
    df_processing_summary = None
else:
    all_clean, all_removed, processing_summary = [], [], []

    for subj in range(1, TOTAL_SUBJECTS + 1):
        try:
            df_clean, df_removed, n_raw = process_subject(subj)
            all_clean.append(df_clean)
            all_removed.append(df_removed)
            processing_summary.append({
                "Subject_ID": subj,
                "Raw_Rows": n_raw,
                "Feature_Cycles": len(df_clean) + len(df_removed),
                "Clean_Cycles": len(df_clean),
                "Removed_Cycles": len(df_removed),
            })
            print(
                f"Subject {subj:02d}/80 ✓  "
                f"| Raw: {n_raw:,}  "
                f"| Cycles: {len(df_clean)+len(df_removed)}  "
                f"| Clean: {len(df_clean)}  "
                f"| Removed: {len(df_removed)}"
            )
        except Exception as e:
            print(f"Subject {subj:02d}/80 ❌  ERROR: {e}")
            processing_summary.append({
                "Subject_ID": subj,
                "Raw_Rows": np.nan,
                "Feature_Cycles": np.nan,
                "Clean_Cycles": np.nan,
                "Removed_Cycles": np.nan,
            })

    df_master = pd.concat(all_clean, ignore_index=True) if all_clean else pd.DataFrame()
    df_processing_summary = pd.DataFrame(processing_summary)

    print("\n" + "=" * 60)
    print("PROCESSING COMPLETE")
    print("=" * 60)
    print("Total clean cycles:", len(df_master))
    print("Subjects retained:", df_master["Subject_ID"].nunique())

## 4. Merge Trial Classification

Attach the subject-level clinical group to every cycle so we can later stratify analyses.

In [ ]:
# Ensure we have subject labels
if "Trial Classification" not in df_master.columns:
    subject_labels = df_info[["Subject Number", "Trial Classification"]].copy()
    subject_labels = subject_labels.rename(columns={"Subject Number": "Subject_ID"})
    df_master = df_master.merge(subject_labels, on="Subject_ID", how="left")

print("Shape after merge:", df_master.shape)
print("Missing Trial Classification:", df_master["Trial Classification"].isnull().sum())
print("\nCycles per Trial Classification:")
print(df_master["Trial Classification"].value_counts())
df_master.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Cycles per subject
cps = df_master.groupby("Subject_ID").size()
axes[0].hist(cps, bins=25, color="#6C8FD9", edgecolor="white")
axes[0].axvline(cps.median(), color="#D94C4C", ls="--", label=f"Median = {cps.median():.0f}")
axes[0].set_xlabel("Clean cycles per subject")
axes[0].set_ylabel("Number of subjects")
axes[0].set_title("Cycles retained per subject")
axes[0].legend()

# Cycles by Trial Classification
tc_cycles = df_master["Trial Classification"].value_counts()
axes[1].barh(tc_cycles.index, tc_cycles.values, color="#5B8DEF")
axes[1].set_xlabel("Number of cycles")
axes[1].set_title("Cycles by Trial Classification")
for i, v in enumerate(tc_cycles.values):
    axes[1].text(v + 30, i, str(v), va="center", fontsize=9)

plt.tight_layout()
plt.savefig("fig_cycles_overview.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nTotal subjects : {df_master['Subject_ID'].nunique()}")
print(f"Total cycles   : {len(df_master):,}")
print(f"Median cycles/subject : {cps.median():.0f}")

## 5. Feature distribution & quality snapshot

Quick sanity check that the extracted features look physiologically reasonable.

In [ ]:
core_features = [
    "Cycle_Duration_s",
    "Flow_Range_L_s",
    "Pressure_Range_cmH2O",
    "Max_Tidal_Volume_L",
    "Chest_Range_mm",
    "Abd_Range_mm",
]

print("Missing values:", df_master.isnull().sum().sum())
print("Duplicate rows:", df_master.duplicated().sum())
print()
display(df_master[core_features].describe().T.round(3))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.ravel()

for ax, feat in zip(axes, core_features):
    ax.hist(df_master[feat].dropna(), bins=40, color="#6C8FD9", edgecolor="white", alpha=0.85)
    ax.set_title(feat, fontsize=11)
    ax.set_xlabel("")
    ax.set_ylabel("Count")

plt.suptitle("Distribution of core respiratory features (after QC)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("fig_feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Technical Correct / Wrong labels

Real inhaler-technique labels (clinician-scored) are not available in this dataset.  
We therefore build a **transparent technical proxy**:

1. Select six key physiological features.
2. Compute the **25th–75th percentile** range on the whole cohort (reference “normal” band).
3. Flag a feature as *abnormal* if it falls outside that band.
4. Count how many features are abnormal for each cycle.
5. Label the cycle **Wrong** if ≥ 2 features are abnormal, otherwise **Correct**.

> **Important for ML:**  
> These six columns (and the derived abnormal flags) must **never** be used as model inputs —  
> they define the label.  Using them would cause perfect but useless leakage.

In [ ]:
label_features = [
    "Cycle_Duration_s",
    "Flow_Range_L_s",
    "Pressure_Range_cmH2O",
    "Max_Tidal_Volume_L",
    "Chest_Range_mm",
    "Abd_Range_mm",
]

print("Features used for technical labeling:")
for f in label_features:
    print("  •", f)

In [ ]:
# Reference inter-quartile ranges
reference = df_master[label_features].describe().loc[["25%", "75%"]]
print("Normal reference ranges (25th–75th percentile):")
display(reference.round(3))

In [ ]:
# Create binary abnormal flags
for feature in label_features:
    low = reference.loc["25%", feature]
    high = reference.loc["75%", feature]
    df_master[feature + "_Abnormal"] = (
        (df_master[feature] < low) | (df_master[feature] > high)
    ).astype(int)

abnormal_cols = [f + "_Abnormal" for f in label_features]
df_master["Abnormal_Features"] = df_master[abnormal_cols].sum(axis=1)

print("Abnormal-feature count distribution:")
print(df_master["Abnormal_Features"].value_counts().sort_index())

In [ ]:
# Final binary technique label
df_master["Technique_Label"] = np.where(
    df_master["Abnormal_Features"] >= 2,
    "Wrong",
    "Correct",
)

label_summary = (
    df_master["Technique_Label"]
    .value_counts()
    .rename_axis("Technique_Label")
    .reset_index(name="Count")
)
label_summary["Percentage"] = (
    label_summary["Count"] / len(df_master) * 100
).round(2)

print("Technical labels created.\n")
display(label_summary)

# Crosstab for transparency
print("\nAbnormal count × Technique Label:")
display(pd.crosstab(df_master["Abnormal_Features"], df_master["Technique_Label"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

colors = ["#5CB85C", "#D94C4C"]
axes[0].bar(
    label_summary["Technique_Label"],
    label_summary["Count"],
    color=colors,
)
axes[0].set_ylabel("Number of cycles")
axes[0].set_title("Correct vs Wrong technique labels")
for i, row in label_summary.iterrows():
    axes[0].text(i, row["Count"] + 150, f"{row['Count']}\n({row['Percentage']}%)",
                 ha="center", fontweight="bold")

abn = df_master["Abnormal_Features"].value_counts().sort_index()
axes[1].bar(abn.index.astype(str), abn.values, color="#6C8FD9")
axes[1].axvline(1.5, color="#D94C4C", ls="--", label="Threshold (≥2 → Wrong)")
axes[1].set_xlabel("Number of abnormal features")
axes[1].set_ylabel("Number of cycles")
axes[1].set_title("Abnormality score distribution")
axes[1].legend()

plt.tight_layout()
plt.savefig("fig_technique_labels.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Final dataset inspection

In [ ]:
print("=" * 60)
print("SMART INHALO MASTER DATASET — FINAL SUMMARY")
print("=" * 60)
print(f"Shape              : {df_master.shape}")
print(f"Subjects           : {df_master['Subject_ID'].nunique()}")
print(f"Total cycles       : {len(df_master):,}")
print(f"Correct cycles     : {(df_master['Technique_Label']=='Correct').sum():,}")
print(f"Wrong cycles       : {(df_master['Technique_Label']=='Wrong').sum():,}")
print(f"Missing values     : {df_master.isnull().sum().sum()}")
print(f"Duplicate rows     : {df_master.duplicated().sum()}")
print()
print("Columns:")
for c in df_master.columns:
    print(f"  • {c}")
print()
df_master.head()

## 8. Save the production dataset

This file is the single source of truth for the downstream ML / DL notebook.

In [ ]:
OUTPUT_FILE = "Smart_Inhalo_Master_Dataset.csv"

df_master.to_csv(OUTPUT_FILE, index=False)

size_mb = os.path.getsize(OUTPUT_FILE) / 1024 / 1024
print(f"Saved → {OUTPUT_FILE}")
print(f"Shape  → {df_master.shape}")
print(f"Size   → {size_mb:.2f} MB")
print()
print("Ready for the Hybrid ML + DL classification notebook.")

## 9. Leakage & design notes (for the presentation)

| Safeguard | Why it matters |
|-----------|----------------|
| **Subject-level train/test split** (done in the ML notebook) | Prevents the model from recognising a person instead of a technique |
| **Label-building columns excluded from features** | The 6 abnormality flags + `Abnormal_Features` define the label; feeding them to the model would give fake perfect accuracy |
| **IQR thresholds computed once on the full cohort** | Transparent, reproducible technical proxy; in a stricter setting thresholds would be fit on train subjects only |
| **QC before labelling** | Non-physiological cycles never enter the label distribution |
| **Bundle model + scaler + feature order** (ML notebook) | Guarantees identical preprocessing at inference time |

### What this dataset is *not*
- It does **not** contain real clinician-scored inhaler technique events.
- The Correct / Wrong label is a **technical proxy** based on signal statistics.

### Natural next step
Replace the technical proxy with real inhaler-use **audio** labels  
(ICBHI 2017 or RDA Drug-Actuation datasets) and train a 1-D CNN / LSTM on the raw waveform.

---

**End of dataset pipeline.**  
Output file: `Smart_Inhalo_Master_Dataset.csv`  
Use it directly in *Smart_Inhalo_Hybrid_ML_DL.ipynb*.